# Continuous Batching 教程

本教程介绍连续批处理的核心概念，这是提升 LLM 推理吞吐量的关键技术。

## 目录
1. 静态批处理的问题
2. 连续批处理原理
3. 调度策略
4. 实战演示

## 1. 静态批处理的问题

传统静态批处理必须等待所有序列完成才能处理下一批：

```
Batch 1: [Seq1: 100 tokens] [Seq2: 50 tokens] [Seq3: 200 tokens]
                            ↑ 等待             ↑ 等待
         |<-------------- 等待最长序列完成 -------------->|
```

**问题**：
- GPU 利用率低 (50-60%)
- 短序列等待长序列
- 吞吐量受限

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.continuous_batch import (
    SchedulerConfig,
    ContinuousBatcher,
    SchedulingPolicy,
    create_continuous_batcher
)

# 模拟静态批处理效率
def static_batch_efficiency(seq_lengths):
    max_len = max(seq_lengths)
    total_compute = sum(seq_lengths)
    wasted_compute = len(seq_lengths) * max_len - total_compute
    return total_compute / (total_compute + wasted_compute)

lengths = [50, 100, 200, 30, 150]
print(f"序列长度: {lengths}")
print(f"静态批处理效率: {static_batch_efficiency(lengths):.1%}")

## 2. 连续批处理原理

连续批处理在每次迭代后动态调整批次：

```
Step 1: [Seq1] [Seq2] [Seq3]
Step 2: [Seq1] [Seq2✓] [Seq3] → Seq2 完成，移出
Step 3: [Seq1] [Seq4] [Seq3] → Seq4 加入
```

**优势**：
- GPU 利用率 >90%
- 吞吐量提升 2-3x
- 延迟优化

In [ ]:
# 创建连续批处理器
config = SchedulerConfig(
    max_batch_size=8,
    max_tokens_per_batch=1024,
    scheduling_policy=SchedulingPolicy.FCFS
)

batcher = ContinuousBatcher(config)
print(f"调度策略: {config.scheduling_policy.value}")
print(f"最大批大小: {config.max_batch_size}")

## 3. 调度策略

支持多种调度策略：
- **FCFS**: 先来先服务
- **SJF**: 最短作业优先
- **Priority**: 优先级调度

In [ ]:
# 添加请求
prompts = [
    "Hello, how are you?",
    "What is machine learning?",
    "Explain quantum computing in simple terms."
]

for i, prompt in enumerate(prompts):
    req_id = batcher.add_request(prompt)
    print(f"添加请求 {req_id}: {prompt[:30]}...")

print(f"\n待处理请求数: {batcher.get_num_pending_requests()}")

In [ ]:
# 调度一批请求
schedule_output = batcher.schedule()

print(f"Prefill 请求数: {len(schedule_output.prefill_requests)}")
print(f"Decode 请求数: {len(schedule_output.decode_requests)}")

## 4. 实战演示

模拟完整的推理流程。

In [ ]:
# 使用工厂函数创建
batcher = create_continuous_batcher(
    max_batch_size=4,
    scheduling_policy="fcfs"
)

# 添加多个请求
for i in range(5):
    batcher.add_request(f"Request {i}: " + "x" * (i * 10 + 10))

# 模拟推理循环
step = 0
while batcher.has_pending_requests() and step < 10:
    output = batcher.schedule()
    total = len(output.prefill_requests) + len(output.decode_requests)
    print(f"Step {step}: 处理 {total} 个请求")
    step += 1

## 总结

连续批处理的核心优势：

1. **高吞吐量**: 2-3x 提升
2. **高 GPU 利用率**: >90%
3. **低延迟**: 短序列不等待
4. **灵活调度**: 支持多种策略

### 参考资料
- [ORCA Paper (OSDI 2022)](https://www.usenix.org/conference/osdi22/presentation/yu)
- [vLLM](https://github.com/vllm-project/vllm)